In [12]:
import os
from dotenv import load_dotenv
from ultralytics.data.converter import convert_coco
import yaml
import glob
import shutil
from ultralytics import YOLO

# force Python to read the .env file 
load_dotenv()

True

In [13]:
# define dataset parent directory
data_dir = os.path.abspath("../data/plates_cropped")

This cell is used to convert, reorganize, and create necessary files to run YOLO models. It expects train and val folders in the ML dataset directory which each contain their own COCO formatted JSON files.

In [14]:
# define the directory containing ML ready tiles
ml_data_dir = os.path.join(data_dir, 'ml_dataset')

# convert the COCO formatted JSONs to YOLO format
print("Converting train data...")
convert_coco(
    labels_dir=os.path.join(ml_data_dir, 'train'), 
    use_segments=True,
    save_dir=os.path.join(ml_data_dir, 'train', 'labels') # FIX: Force output location
)

print("Converting val data...")
convert_coco(
    labels_dir=os.path.join(ml_data_dir, 'val'), 
    use_segments=True,
    save_dir=os.path.join(ml_data_dir, 'val', 'labels') # FIX: Force output location
)
print("Converted files to YOLO format\n")

# reorganize the label files into the correct structure for YOLO
print("Reorganizing label files into correct YOLO folders...")

for split in ['train', 'val']:
    labels_target_dir = os.path.join(ml_data_dir, split, "labels")
    os.makedirs(labels_target_dir, exist_ok=True)
    
    # search for all .txt files anywhere inside the split folder
    all_txts = glob.glob(os.path.join(ml_data_dir, split, "**", "*.txt"), recursive=True)
    
    moved_count = 0
    for txt_path in all_txts:
        # ignore YOLO's classes summary file
        if os.path.basename(txt_path) == "classes.txt":
            continue
            
        # move it if it isn't already in the target 'labels' folder
        if os.path.dirname(txt_path) != os.path.abspath(labels_target_dir):
            shutil.move(txt_path, os.path.join(labels_target_dir, os.path.basename(txt_path)))
            moved_count += 1
            
    print(f"Moved {moved_count} label files to {split}/labels/")

# create YOLO .yaml configuration file
yolo_yaml_path = os.path.join(ml_data_dir, "chaos_blocks.yaml")

yolo_config = {
    # provide the path to the dataset
    "path": os.path.abspath(ml_data_dir),
    
    # YOLO looks for a 'labels' folder next to these 'images' folders
    "train": "train/images",
    "val": "val/images",
    
    # class dictionary
    "names": {
        0: "chaos_block"
    }
}

with open(yolo_yaml_path, 'w') as f:
    yaml.dump(yolo_config, f, default_flow_style=False)

print(f"Created YOLO config file at: {yolo_yaml_path}")

Converting train data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/train/train_coco.json: 100% ━━━━━━━━━━━━ 1176/1176 2.7Kit/s 0.4s0.1s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/train/labels
Converting val data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/val/val_coco.json: 100% ━━━━━━━━━━━━ 42/42 3.3Kit/s 0.0s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/val/labels
Converted files to YOLO format

Reorganizing label files into correct YOLO folders...
Moved 1176 label files to train/labels/
Moved 42 label files to val/labels/
Created YOLO config file at: /Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/chaos_blocks.yaml


In [15]:
# define absolute paths to the yaml file and run results directory
# define the directory containing ML ready tiles
ml_data_dir = os.path.join(data_dir, 'ml_dataset')
yaml_path = ml_data_dir = os.path.join(ml_data_dir, 'chaos_blocks.yaml')
results_dir = os.path.abspath("../results/runs")

# define where you want the base model to be saved/loaded from
model_dir = os.path.abspath("../models/")
os.makedirs(model_dir, exist_ok=True)

# using the pretrained model here
model_path = os.path.join(model_dir, "yolo26s-seg.pt")

print(f"Base model location: {model_path}")
print(f"Saving runs to: {results_dir}")

# load the model
model = YOLO(model_path) 

results = model.train(
    data=yaml_path,
    epochs=50,
    patience=15,  # if mAP doesn't improve for 15 epochs, stop early
    imgsz=512,    # up sample the image to 512
    batch=8,      # decrease batch size to compensate for upsampled image
    device='mps',           # hardware for running
    project=results_dir,    # define directory to store the results
    name="run_16_yolo26s",  # folder to save results
    #lr0=0.001,           # Small, careful learning steps
    #cos_lr=True,         # Smooth learning curve
    #weight_decay=0.001,  # Strict penalty for overfitting
    #pretrained=False,
    overlap_mask=False,     # False means model will not automatically merge overlapping boxes
    iou=0.7,                # IoU threshold (loswer = more aggressive suppression)
    box=7.5,                # Multiplier for bounding box loss 
    close_mosaic=0,     
    #hsv_h=0.0,           # Disable Hue shifts (Keep it grayscale)
    #hsv_s=0.0,           # Disable Saturation shifts
    #hsv_v=0.6,           # Aggressive Brightness shifts (Shadow resilience)
    #bgr=0.2              # 20% chance of Blur
)

Base model location: /Users/jjwill18/Desktop/europa-chaos-blocks/models/yolo26s-seg.pt
Saving runs to: /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs
New https://pypi.org/project/ultralytics/8.4.182 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.11.16 torch-2.13.0 MPS (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/jjwill18/Desktop/europa-chaos-blocks/data/plates_cropped/ml_dataset/chaos_blocks.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      7.77G      2.044      3.384      2.924   0.009675      3.705         40        512: 100% ━━━━━━━━━━━━ 150/150 2.1it/s 1:130.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6it/s 1.9s1.2s
                   all         42        432      0.167      0.266      0.101     0.0423      0.168      0.259     0.0983     0.0305

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size


/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      4.79G      2.031      3.204      2.377   0.009252      3.278          7        512: 100% ━━━━━━━━━━━━ 150/150 2.2it/s 1:090.5s8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6it/s 1.9s1.2s
                   all         42        432      0.111       0.13     0.0417     0.0157      0.115       0.12     0.0377     0.0115

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size


/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (81 outputs left out)

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      44/50      4.74G       1.49      2.086      1.021    0.00621      1.145         10        512: 100% ━━━━━━━━━━━━ 150/150 2.3it/s 1:040.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.8it/s 1.6s1.1s
                   all         42        432      0.198      0.216      0.116     0.0436      0.171      0.213     0.0958     0.0348

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      45/50      4.65G      1.637      2.039     0.9333   0.006248     0.9841        169        512: 0% ──────────── 0/150  1.0s

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      45/50      4.74G      1.453      2.029     0.9604   0.006038       1.11         68        512: 100% ━━━━━━━━━━━━ 150/150 6.5s/it 16:220.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.8s1.1s
                   all         42        432      0.195      0.231      0.105     0.0446      0.185      0.225     0.0933      0.035
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 30, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

45 epochs completed in 1.247 hours.
Optimizer stripped from /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs/run_16_yolo26s/weights/last.pt, 23.3MB
Optimizer stripped from /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs/run_16_yolo26s/weights/best.pt, 